# Phân tích baseline v4
Đọc output thực nghiệm v4. `score_proxy` là metric xấp xỉ; `state_hash_match` kiểm trạng thái cuối. `done` có thể là quyết định planner hoặc code đã chạy hết plan có after=done; chưa bảo đảm task hoàn tất.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path("/kaggle/working/lifemate_baseline_v4")  # đổi thành thư mục report đã tải
if not ROOT.exists():
    ROOT = Path("lifemate_baseline_v4")

def jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()] if path.exists() else []

def read_run(mode):
    base = ROOT / mode
    return {
        "summary": json.loads((base/"summary.json").read_text()) if (base/"summary.json").exists() else {},
        "cases": pd.DataFrame(jsonl(base/"traces.jsonl")),
        "metrics": pd.read_csv(base/"cases.csv") if (base/"cases.csv").exists() and (base/"cases.csv").stat().st_size else pd.DataFrame(),
        "status": json.loads((base/"run_status.json").read_text()) if (base/"run_status.json").exists() else {},
    }
runs = {mode: read_run(mode) for mode in ("workflow_gold", "baseline_asr")}
for mode, run in runs.items():
    print(mode, run["status"])
    display(run["summary"])


## Metric, lỗi và điểm dừng
`invalid_model_output`: vi phạm contract, không có vòng tự sửa bằng feedback của Qwen. `validation_failed`/`tool_failed`: fixed mode dừng khi env báo lỗi. `validation_no_progress`: lặp lỗi env.validate ở adaptive mode. `environment_no_progress`: tool thất bại hoặc read lặp. `duplicate_write`: code dừng trước khi lặp write thành công.


In [ ]:
for mode, run in runs.items():
    df = run["metrics"]
    if df.empty:
        continue
    print(mode)
    display(df[[c for c in ["id", "score_proxy", "state_hash_match", "stop_reason", "tool_count", "planner_calls", "replans", "execution_steps", "agent_latency_s"] if c in df]])
    display(df.stop_reason.value_counts().rename("count").to_frame())
    display(df.loc[df.score_proxy < 1, ["id", "error_flags", "missing_tool_names", "extra_tool_names"]])


## Chi phí từng node
Trace chứa `node_outputs` cho `plan`, `select_arguments`, `fill`. Node bị skip không phát sinh request. Các batch gom request của nhiều mẫu, không ghép dữ liệu mẫu vào cùng một prompt.


In [ ]:
node_rows = []
for mode, run in runs.items():
    for case in run["cases"].to_dict("records"):
        for event in case.get("trace", []):
            for node, output in event.get("node_outputs", {}).items():
                node_rows.append({"mode": mode, "id": case["id"], "decision": event["decision"], "node": node,
                    **{k: output.get(k) for k in ["prompt_tokens", "generation_tokens", "latency_s"]}})
nodes = pd.DataFrame(node_rows)
if not nodes.empty:
    display(nodes.groupby(["mode", "node"])[["prompt_tokens", "generation_tokens", "latency_s"]].agg(["count", "mean", "max"]))


In [ ]:
def show_case(sample_id, mode="workflow_gold"):
    run = runs[mode]
    traces = run["cases"]
    if traces.empty or sample_id not in set(traces.id):
        print("Không tìm thấy", sample_id)
        return
    metric = run["metrics"]
    if not metric.empty:
        display(metric.loc[metric.id == sample_id].T)
    case = traces.loc[traces.id == sample_id].iloc[0].to_dict()
    print(json.dumps(case, ensure_ascii=False, indent=2))

# show_case("<sample_id>")


## Diễn giải
- Đối chiếu request với tool calls để phát hiện done sớm, thiếu action, chọn sai tool/field/value.
- `validation.ok=true` chỉ xác nhận schema, không xác nhận ID hay nội dung đúng ý người dùng.
- `tool_result.ok=true, done=false` là thất bại nghiệp vụ; xem response thật được chuyển vào lượt tiếp theo.
- `local_diagnostic` chỉ là log code; không nằm trong env_feedback gửi model.
- So tốc độ bằng `experiment_wall_s`/`samples_per_second`. Với workers > 1, latency mỗi request gồm thời gian chờ batch. Chạy workers=1 để đo latency riêng.
